# Figure S2 — Quantitative LinkD-Bind benchmarking

RMSE and Pearson for representative methods across datasets and splits.

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

- `source_data/TableS2_Benchmarking_LinkD.xlsx`

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Panels — RMSE and Pearson grids

In [ ]:

df = io.read_table_s2()
test = df[df["Dataset"] == "Test"].copy()
methods = ["LinkD", "Diffusion", "DeepDTA", "DeepPurpose", "GraphDTA"]
test = test[test["Model"].isin(methods)]
metrics = ["RMSE", "Pearson"]
modes = ["random", "cold_protein", "cold_drug"]
datasets = ["BindDB", "Davis", "Kiba"]

fig, axes = plt.subplots(2, 3, figsize=(9, 5.5), sharey=False)
for col, mode in enumerate(modes):
    for row, metric in enumerate(metrics):
        ax = axes[row, col]
        sub = test[test["Mode"] == mode]
        piv = sub.pivot_table(index="Model", columns="Data", values=metric, aggfunc="mean")
        piv = piv.reindex(methods)
        piv = piv[[d for d in datasets if d in piv.columns]]
        piv.plot(kind="bar", ax=ax, width=0.8, legend=(row==0 and col==2))
        ax.set_title(f"{metric} | {mode}")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45)
        if col == 2 and row == 0:
            ax.legend(frameon=False, fontsize=6, title="Dataset")
fig.suptitle("Fig S2 — test-set RMSE / Pearson", y=1.02)
fig.tight_layout()
out = style.save_panel(fig, "figS2_rmse_pearson", test)
plt.show()
print(out)
